# Machine Learning Classification of Variable Stars Using Gaia DR3 Light Curves

## Imports and Configuration

In [1]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for path in (root / "src", root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gaia_variable_stars.config import (
    CLASSES, CV_FILE, FEATURE_FILE, FIGURES_ROOT, FOLDS,
    MIN_POINTS, MODEL_FILE, MODELS_ROOT, PRED_FILE,
    RANK_FILE, SEED, SOURCE_FILE,
)

FE_FEATURE_FILE = FEATURE_FILE.with_name("gaia_features_fe.csv")
FE_VALIDATION_FILE = FEATURE_FILE.with_name("gaia_period_validation_fe.csv")
FE_ORACLE_FILE = FEATURE_FILE.with_name("gaia_features_fe_oracle.csv")

EXTRA_ROOT = MODELS_ROOT / "gaia_classification"
PLOT_ROOT = FIGURES_ROOT / "gaia_classification"

for path in (MODELS_ROOT, EXTRA_ROOT, PLOT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

REPEATS = 5
BOOTSTRAPS = 1500
CLUSTER_DRAWS = 300
PERM_REPEATS = 5
MIN_CELL = 10

LMC_CENTER = (80.8942, -69.7561)
SMC_CENTER = (13.1867, -72.8286)
LMC_RADIUS = 8.0
SMC_RADIUS = 5.0

## Features

In [2]:
GROUPS = {
    "period": [
        "log_period", "power", "half_power", "double_power",
        "half_power_ratio", "double_power_ratio", "fourier_amp", "fourier_ratio",
    ],
    "shape": [
        "magnitude_std", "magnitude_skew", "magnitude_kurt",
        "magnitude_mad", "amplitude", "beyond_1sigma_fraction",
    ],
    "timescale": [
        "von_neumann_eta", "stetson_j", "sf_short",
        "sf_long", "sf_ratio", "max_visit_slope",
    ],
    "color": ["bp_rp_color"],
    "brightness_noise": ["magnitude_mean", "median_error", "amplitude_ratio"],
    "sampling": [
        "n_epochs", "time_span", "median_gap", "short_gap_fraction",
        "n_visits", "epochs_per_visit", "median_visit_gap", "visit_gap_cv",
    ],
}

MAIN_GROUPS = [name for name in GROUPS if name != "sampling"]

def columns(names):
    return [col for name in names for col in GROUPS[name]]

BASE_FEATURES = columns(MAIN_GROUPS)
ALL_FEATURES = columns(GROUPS)
INTRINSIC_FEATURES = [
    col for col in BASE_FEATURES
    if col not in {"magnitude_mean", "median_error", "amplitude_ratio"}
]

FEATURE_SETS = {
    "full": BASE_FEATURES,
    "oracle": BASE_FEATURES,
    "period_free": columns([name for name in MAIN_GROUPS if name != "period"]),
    "intrinsic": INTRINSIC_FEATURES,
    "full_plus_sampling": ALL_FEATURES,
    "color_amp": ["bp_rp_color", "amplitude"],
    "no_indicators": BASE_FEATURES,
}

for name in GROUPS:
    FEATURE_SETS[f"leave_{name}"] = columns([g for g in MAIN_GROUPS if g != name])
    FEATURE_SETS[f"add_{name}"] = GROUPS[name]

## Data

In [3]:
sources = pd.read_csv(SOURCE_FILE)
features = pd.read_csv(FE_FEATURE_FILE)
validation = pd.read_csv(FE_VALIDATION_FILE)
oracle = pd.read_csv(FE_ORACLE_FILE)

source_cols = [
    "source_id", "class_name", "ra", "dec", "ecl_lat",
    "bp_rp_color", "g_mean_magnitude", "gaia_period", "gaia_period_origin",
]

period_cols = [
    "period_eligible", "phase_coherent_recovery",
    "relative_period_accurate", "half_period_alias",
    "double_period_alias", "n_cycles", "sky_cell",
]

feature_cols = columns(["period", "shape", "timescale", "brightness_noise"])

stars = (
    sources[source_cols]
    .merge(
        validation[["source_id", *period_cols, *GROUPS["sampling"]]],
        on="source_id", how="left", validate="one_to_one"
    )
    .merge(
        features[["source_id", *feature_cols]],
        on="source_id", how="left", validate="one_to_one"
    )
    .sort_values("source_id")
    .reset_index(drop=True)
)

if len(stars) != 2000 or stars["sky_cell"].isna().any():
    raise ValueError("Expected 2,000 stars with period-validation and sky-cell data.")

dropped = len(stars) - len(features)
stars = stars.loc[stars["source_id"].isin(features["source_id"])].reset_index(drop=True)
assert len(stars) == len(features)

numeric = list(dict.fromkeys([
    *ALL_FEATURES, "ra", "dec", "ecl_lat", "g_mean_magnitude",
    "gaia_period", "n_cycles",
]))

stars[numeric] = stars[numeric].apply(pd.to_numeric, errors="coerce")
stars[numeric] = stars[numeric].replace([np.inf, -np.inf], np.nan)

stars["eligible"] = stars["period_eligible"].fillna(False).astype(bool)
stars["period_recovered"] = (
    stars["eligible"] & stars["phase_coherent_recovery"].fillna(False).astype(bool)
)

stars["period_status"] = np.select(
    [~stars["eligible"], stars["period_recovered"]],
    ["ineligible", "recovered"],
    default="mismatched",
)

stars["abs_ecl_lat"] = stars["ecl_lat"].abs()

oracle_stars = stars.copy()
reference = oracle.set_index("source_id").reindex(
    stars.loc[stars["eligible"], "source_id"]
)

if reference["log_period"].isna().any():
    raise ValueError("Oracle periods are missing for eligible stars.")

oracle_stars.loc[stars["eligible"], GROUPS["period"]] = (
    reference[GROUPS["period"]].to_numpy()
)

print("Total stars:", len(stars))
print("Feature rows:", len(features))
print("Rows without extracted features:", dropped)

display(
    pd.crosstab(stars["class_name"], stars["period_status"])
    .reindex(
        index=CLASSES,
        columns=["recovered", "mismatched", "ineligible"],
        fill_value=0,
    )
)

Total stars: 1978
Feature rows: 1978
Rows without extracted features: 22


period_status,recovered,mismatched,ineligible
class_name,,,
RR Lyrae,392,86,0
Cepheid,495,5,0
Eclipsing Binary,12,488,0
Long-Period Variable,269,132,99


## Cross-Validation

In [4]:
y = stars["class_name"].to_numpy()
sky = stars["sky_cell"].to_numpy()

splits = []
folds = np.zeros((REPEATS, len(stars)), dtype=np.uint8)

for repeat in range(REPEATS):
    cv = StratifiedGroupKFold(
        n_splits=FOLDS, shuffle=True, random_state=SEED + 101 * repeat
    )

    for fold, (train, test) in enumerate(cv.split(stars, y, sky), start=1):
        if len(np.unique(y[train])) != len(CLASSES):
            raise ValueError(f"Training class missing in repeat {repeat + 1}, fold {fold}.")

        folds[repeat, test] = fold
        splits.append((repeat, fold, train, test))

if (folds == 0).any():
    raise ValueError("Some stars have no out-of-fold prediction assignment.")

print("Sky cells:", stars["sky_cell"].nunique())
print("Folds:", FOLDS)
print("Repeats:", REPEATS)

Sky cells: 167
Folds: 5
Repeats: 5


## Models

In [5]:
def make_model(name, variant):
    if name == "RF":
        return Pipeline([
            ("imputer", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True
            )),
            ("model", RandomForestClassifier(
                n_estimators=250, min_samples_leaf=2,
                max_features="sqrt", random_state=SEED, n_jobs=-1
            )),
        ])

    model = HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, min_samples_leaf=15,
        l2_regularization=1.0, random_state=SEED,
    )

    if variant == "no_indicators":
        return model

    return Pipeline([
        ("flags", FeatureUnion([
            ("values", FunctionTransformer()),
            ("missing", MissingIndicator(features="all", sparse=False)),
        ])),
        ("model", model),
    ])

In [6]:
importance_rows = []

def predict_oof(name, variant):
    cols = FEATURE_SETS[variant]
    data = oracle_stars if variant == "oracle" else stars
    result = np.full((REPEATS, len(stars)), None, dtype=object)

    for repeat, fold, train, test in splits:
        model = make_model(name, variant)
        model.fit(data.iloc[train][cols], y[train])
        result[repeat, test] = model.predict(data.iloc[test][cols])

        if name == "HGB" and variant == "full" and repeat == 0:
            score = permutation_importance(
                model, data.iloc[test][cols], y[test],
                scoring=lambda fit, x, truth: f1_score(
                    truth, fit.predict(x), labels=CLASSES,
                    average="macro", zero_division=0
                ),
                n_repeats=PERM_REPEATS, random_state=SEED + fold, n_jobs=1,
            )

            for col, decrease in zip(cols, score.importances_mean):
                importance_rows.append({
                    "fold": fold, "feature": col,
                    "macro_f1_drop": float(decrease),
                })

    return result

## RF and HGB Predictions

In [7]:
MAIN = [
    "full", "oracle", "period_free",
    "intrinsic", "full_plus_sampling", "color_amp",
]

predictions = {}

for variant in MAIN[:3]:
    predictions[("RF", variant)] = predict_oof("RF", variant)
    print("RF:", variant)

RF: full
RF: oracle
RF: period_free


In [8]:
for variant in MAIN[3:]:
    predictions[("RF", variant)] = predict_oof("RF", variant)
    print("RF:", variant)

RF: intrinsic
RF: full_plus_sampling
RF: color_amp


In [9]:
for variant in MAIN[:3]:
    predictions[("HGB", variant)] = predict_oof("HGB", variant)
    print("HGB:", variant)

HGB: full
HGB: oracle
HGB: period_free


In [10]:
for variant in [*MAIN[3:], "no_indicators"]:
    predictions[("HGB", variant)] = predict_oof("HGB", variant)
    print("HGB:", variant)

HGB: intrinsic
HGB: full_plus_sampling
HGB: color_amp
HGB: no_indicators


## Group Ablations

In [11]:
for name in GROUPS:
    variant = f"leave_{name}"

    if name == "sampling":
        predictions[("HGB", variant)] = predictions[("HGB", "full")].copy()
    else:
        predictions[("HGB", variant)] = predict_oof("HGB", variant)

    print("Leave out:", name)

Leave out: period
Leave out: shape
Leave out: timescale
Leave out: color
Leave out: brightness_noise
Leave out: sampling


In [ ]:
for name in GROUPS:
    variant = f"add_{name}"
    predictions[("HGB", variant)] = predict_oof("HGB", variant)
    print("Add only:", name)

Add only: period
Add only: shape
Add only: timescale
Add only: color


## Out-of-Fold Predictions

In [ ]:
oof = pd.concat([
    stars[["source_id", "class_name", "sky_cell", "period_status"]]
    .assign(repeat=repeat + 1, fold=folds[repeat])
    for repeat in range(REPEATS)
], ignore_index=True)

for (name, variant), values in predictions.items():
    pred_col = f"pred_{name}_{variant}"
    correct_col = f"correct_{name}_{variant}"

    oof[pred_col] = values.ravel()
    oof[correct_col] = oof[pred_col].eq(oof["class_name"])

if len(oof) != len(stars) * REPEATS or oof.filter(like="pred_").isna().any().any():
    raise ValueError("Out-of-fold predictions are incomplete.")

oof.to_csv(PRED_FILE, index=False)

print("Unique stars:", oof["source_id"].nunique())
print("Out-of-fold predictions per model:", len(oof))
print("Feature sets:", len(predictions))

## Model Scores

In [ ]:
score_rows = []
class_rows = []

for (name, variant) in predictions:
    col = f"pred_{name}_{variant}"

    for repeat, part in oof.groupby("repeat"):
        truth = part["class_name"]
        predicted = part[col]

        score_rows.append({
            "model": name,
            "feature_set": variant,
            "repeat": repeat,
            "n": len(part),
            "accuracy": accuracy_score(truth, predicted),
            "balanced_accuracy": balanced_accuracy_score(truth, predicted),
            "macro_f1": f1_score(
                truth, predicted, labels=CLASSES,
                average="macro", zero_division=0
            ),
        })

        for class_name in CLASSES:
            group = part.loc[truth.eq(class_name)]
            class_rows.append({
                "model": name, "feature_set": variant,
                "repeat": repeat, "class_name": class_name,
                "n": len(group), "recall": group[col].eq(class_name).mean(),
            })

cv_scores = pd.DataFrame(score_rows)
class_scores = pd.DataFrame(class_rows)

rank_rows = []

for (name, variant), part in cv_scores.groupby(["model", "feature_set"]):
    row = {"model": name, "feature_set": variant, "repeats": REPEATS}

    for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
        values = part[metric].to_numpy()
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_min"] = values.min()
        row[f"{metric}_max"] = values.max()

    rank_rows.append(row)

cv_ranking = pd.DataFrame(rank_rows).sort_values(["feature_set", "model"])

cv_class = class_scores.groupby(
    ["model", "feature_set", "class_name"], as_index=False
).agg(
    recall_mean=("recall", "mean"),
    recall_min=("recall", "min"),
    recall_max=("recall", "max"),
)

cv_scores.to_csv(CV_FILE, index=False)
cv_ranking.to_csv(RANK_FILE, index=False)
cv_class.to_csv(EXTRA_ROOT / "classes.csv", index=False)

selected = cv_ranking.loc[cv_ranking["feature_set"].isin(MAIN)]

display(
    selected.pivot(
        index="feature_set", columns="model",
        values=[
            "accuracy_mean", "accuracy_min", "accuracy_max",
            "macro_f1_mean", "macro_f1_min", "macro_f1_max",
        ],
    ).round(4)
)

## Paired Bootstrap

In [ ]:
def bootstrap(part, first, second=None, seed=SEED):
    cols = [first] + ([second] if second else [])
    average = part.groupby("source_id")[cols].mean()

    values = (
        average[second] - average[first]
        if second else average[first]
    ).to_numpy(dtype=float)

    n = len(values)

    if n == 0:
        return 0, np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    draws = rng.integers(n, size=(BOOTSTRAPS, n))
    means = values[draws].mean(axis=1)

    return (
        n, float(values.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )

## Ablation Results

In [ ]:
ablation_rows = []

for name in GROUPS:
    for mode in ["leave", "add"]:
        variant = f"{mode}_{name}"
        correct = f"correct_HGB_{variant}"

        n, accuracy, acc_lo, acc_hi = bootstrap(
            oof, correct, seed=SEED + len(ablation_rows)
        )

        delta, delta_lo, delta_hi = np.nan, np.nan, np.nan

        # A negative leave-minus-full delta means the group helped.
        if mode == "leave" and name != "sampling":
            _, delta, delta_lo, delta_hi = bootstrap(
                oof, "correct_HGB_full", correct,
                seed=SEED + len(ablation_rows),
            )

        score = cv_ranking.loc[
            cv_ranking["model"].eq("HGB")
            & cv_ranking["feature_set"].eq(variant)
        ].iloc[0]

        ablation_rows.append({
            "mode": mode, "group": name,
            "n_stars": n, "accuracy": accuracy,
            "accuracy_lo": acc_lo, "accuracy_hi": acc_hi,
            "macro_f1": score["macro_f1_mean"],
            "delta_leave_minus_full": delta,
            "delta_lo": delta_lo, "delta_hi": delta_hi,
        })

ablations = pd.DataFrame(ablation_rows)
ablations.to_csv(EXTRA_ROOT / "ablation.csv", index=False)

display(ablations.round(4))

## Sampling Check

In [ ]:
sampling_rows = []

for name in ["RF", "HGB"]:
    before = f"correct_{name}_full"
    after = f"correct_{name}_full_plus_sampling"

    n, delta, low, high = bootstrap(oof, before, after)
    paired = oof.groupby("source_id")[[before, after]].mean()

    sampling_rows.append({
        "model": name, "n_stars": n,
        "accuracy_delta": delta,
        "ci_lo": low, "ci_hi": high,
        "stars_helped": int((paired[after] > paired[before]).sum()),
        "stars_hurt": int((paired[after] < paired[before]).sum()),
    })

sampling_check = pd.DataFrame(sampling_rows)
sampling_check.to_csv(EXTRA_ROOT / "sampling.csv", index=False)

display(sampling_check.round(4))

## Missing Values

In [ ]:
missing_rows = []

for col in ALL_FEATURES:
    missing = stars[col].isna()
    total_missing = missing.sum()

    for name in CLASSES:
        in_class = stars["class_name"].eq(name)
        count = int((missing & in_class).sum())

        missing_rows.append({
            "feature": col, "class_name": name,
            "missing": count,
            "class_size": int(in_class.sum()),
            "class_missing_pct": 100 * missing[in_class].mean(),
            "missing_from_class_pct": (
                100 * count / total_missing if total_missing else np.nan
            ),
        })

missing_check = pd.DataFrame(missing_rows)
missing_check.to_csv(EXTRA_ROOT / "missing.csv", index=False)

display(
    missing_check.loc[missing_check["feature"].eq("bp_rp_color")]
    .round(2)
)

In [ ]:
before = "correct_HGB_full"
after = "correct_HGB_no_indicators"

n, delta, low, high = bootstrap(oof, before, after, seed=SEED + 908)
paired = oof.groupby("source_id")[[before, after]].mean()

indicator_check = pd.DataFrame([{
    "model": "HGB", "comparison": "native_NaN_minus_explicit_flags",
    "n_stars": n, "accuracy_delta": delta,
    "ci_lo": low, "ci_hi": high,
    "stars_helped": int((paired[after] > paired[before]).sum()),
    "stars_hurt": int((paired[after] < paired[before]).sum()),
}])

indicator_check.to_csv(EXTRA_ROOT / "indicators.csv", index=False)

display(indicator_check.round(4))

## Period Status

In [ ]:
status_rows = []
paired_rows = []

STATUSES = ["recovered", "mismatched", "ineligible"]
PERIOD_SETS = ["full", "oracle", "period_free", "color_amp"]

for name in ["RF", "HGB"]:
    for class_name in CLASSES:
        for status in STATUSES:
            part = oof.loc[
                oof["class_name"].eq(class_name)
                & oof["period_status"].eq(status)
            ]

            if part.empty:
                continue

            variants = [
                variant for variant in PERIOD_SETS
                if status != "ineligible" or variant != "oracle"
            ]

            for variant in variants:
                n, recall, low, high = bootstrap(
                    part, f"correct_{name}_{variant}",
                    seed=SEED + 100 * CLASSES.index(class_name)
                    + 10 * STATUSES.index(status) + PERIOD_SETS.index(variant),
                )

                status_rows.append({
                    "model": name, "class_name": class_name,
                    "period_status": status, "feature_set": variant,
                    "n_stars": n, "recall": recall,
                    "ci_lo": low, "ci_hi": high,
                    "small_cell": n < MIN_CELL,
                })

            pairs = [
                ("full", "period_free"),
                ("full", "color_amp"),
            ]

            if status != "ineligible":
                pairs += [
                    ("full", "oracle"),
                    ("oracle", "period_free"),
                ]

            for first, second in pairs:
                n, delta, low, high = bootstrap(
                    part, f"correct_{name}_{first}",
                    f"correct_{name}_{second}",
                    seed=SEED + 100 * CLASSES.index(class_name)
                    + 10 * STATUSES.index(status) + PERIOD_SETS.index(second),
                )

                paired_rows.append({
                    "model": name, "class_name": class_name,
                    "period_status": status,
                    "reference": first, "compared": second,
                    "n_stars": n, "recall_delta": delta,
                    "ci_lo": low, "ci_hi": high,
                    "small_cell": n < MIN_CELL,
                })

status_scores = pd.DataFrame(status_rows)
paired_status = pd.DataFrame(paired_rows)

status_scores.to_csv(EXTRA_ROOT / "status.csv", index=False)
paired_status.to_csv(EXTRA_ROOT / "paired.csv", index=False)

display(
    status_scores.loc[status_scores["period_status"].eq("mismatched")]
    .round(3)
)

display(
    paired_status.loc[paired_status["period_status"].eq("mismatched")]
    .round(3)
)

## Field Test

In [ ]:
def distance(ra, dec, center):
    ra0, dec0 = np.deg2rad(center)
    alpha = np.deg2rad(np.asarray(ra, dtype=float))
    delta = np.deg2rad(np.asarray(dec, dtype=float))

    cosine = (
        np.sin(delta) * np.sin(dec0)
        + np.cos(delta) * np.cos(dec0) * np.cos(alpha - ra0)
    )

    return np.rad2deg(np.arccos(np.clip(cosine, -1, 1)))


lmc = distance(stars["ra"], stars["dec"], LMC_CENTER) <= LMC_RADIUS
smc = distance(stars["ra"], stars["dec"], SMC_CENTER) <= SMC_RADIUS

stars["field"] = np.select([lmc, smc], ["LMC", "SMC"], default="outside")

train = np.flatnonzero(stars["field"].eq("outside"))
test = np.flatnonzero(stars["field"].ne("outside"))

field_rows = []
field_preds = []

for name in ["RF", "HGB"]:
    for variant in ["full", "intrinsic", "period_free", "color_amp"]:
        cols = FEATURE_SETS[variant]
        model = make_model(name, variant)

        model.fit(stars.iloc[train][cols], y[train])
        predicted = model.predict(stars.iloc[test][cols])

        heldout = stars.iloc[test][["source_id", "class_name", "field"]].copy()
        heldout["predicted"] = predicted
        heldout["model"] = name
        heldout["feature_set"] = variant
        field_preds.append(heldout)

        for field in ["LMC", "SMC", "combined"]:
            part = heldout if field == "combined" else heldout.loc[heldout["field"].eq(field)]

            for class_name in CLASSES:
                group = part.loc[part["class_name"].eq(class_name)]

                field_rows.append({
                    "model": name, "feature_set": variant,
                    "field": field, "class_name": class_name,
                    "n_train": len(train), "n_test": len(group),
                    "recall": group["predicted"].eq(class_name).mean() if len(group) else np.nan,
                })

field_scores = pd.DataFrame(field_rows)
field_preds = pd.concat(field_preds, ignore_index=True)

field_scores.to_csv(EXTRA_ROOT / "field.csv", index=False)
field_preds.to_csv(EXTRA_ROOT / "field_preds.csv", index=False)

print("LMC:", int(lmc.sum()))
print("SMC:", int(smc.sum()))
print("Training outside both:", len(train))

display(
    field_scores.loc[field_scores["field"].eq("combined")]
    .round(3)
)

## Period Recovery

In [ ]:
def wilson(successes, n):
    if n == 0:
        return np.nan, np.nan

    z = norm.ppf(0.975)
    rate = successes / n
    denominator = 1 + z**2 / n
    center = (rate + z**2 / (2 * n)) / denominator
    half = z * np.sqrt(
        rate * (1 - rate) / n + z**2 / (4 * n**2)
    ) / denominator

    return max(0, center - half), min(1, center + half)

In [ ]:
stars["epoch_bin"] = pd.cut(
    stars["n_epochs"], [MIN_POINTS - 1, 30, 50, 80, np.inf],
    labels=["20-30", "31-50", "51-80", "81+"],
)

stars["latitude_bin"] = pd.cut(
    stars["abs_ecl_lat"], [0, 30, 60, 90],
    labels=["0-30", "30-60", "60-90"],
    include_lowest=True,
)

def recovery_table(names):
    rows = []

    for key, group in stars.loc[stars["eligible"]].groupby(names, observed=True):
        key = key if isinstance(key, tuple) else (key,)
        n = len(group)
        recovered = int(group["period_recovered"].sum())
        low, high = wilson(recovered, n)

        rows.append({
            **dict(zip(names, map(str, key))),
            "eligible": n,
            "recovered": recovered,
            "recovery_rate": recovered / n,
            "ci_lo": low, "ci_hi": high,
            "relative_10pct": group["relative_period_accurate"].mean(),
            "small_cell": n < MIN_CELL,
        })

    return pd.DataFrame(rows)


recovery_class = recovery_table(["class_name"])
recovery_epoch = recovery_table(["class_name", "epoch_bin"])
recovery_latitude = recovery_table(["class_name", "latitude_bin"])
recovery_joint = recovery_table(["class_name", "epoch_bin", "latitude_bin"])

recovery_class.to_csv(EXTRA_ROOT / "recovery.csv", index=False)
recovery_epoch.to_csv(EXTRA_ROOT / "epochs.csv", index=False)
recovery_latitude.to_csv(EXTRA_ROOT / "latitude.csv", index=False)
recovery_joint.to_csv(EXTRA_ROOT / "joint.csv", index=False)

display(recovery_class.round(3))
display(recovery_epoch.round(3))
display(recovery_latitude.round(3))
display(recovery_joint.round(3))

## Recovery Regression

In [ ]:
RECOVERY_CLASSES = ["RR Lyrae", "Long-Period Variable"]

def recovery_design(group):
    x = pd.DataFrame({
        "log2_visits": np.log2(group["n_visits"].where(group["n_visits"].gt(0))),
        "latitude_30deg": group["abs_ecl_lat"] / 30,
        "log10_amp_ratio": np.log10(
            group["amplitude_ratio"].where(group["amplitude_ratio"].gt(0))
        ),
        "g_mag": group["g_mean_magnitude"],
    }, index=group.index)

    origin = pd.get_dummies(
        group["gaia_period_origin"].fillna("unknown"),
        prefix="origin", drop_first=True, dtype=float,
    )

    return pd.concat([x, origin], axis=1).replace([np.inf, -np.inf], np.nan)


def fit_recovery(x, target):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(x)

    model = LogisticRegression(
        C=10.0, max_iter=2000, random_state=SEED
    ).fit(scaled, target)

    return model.coef_[0] / scaler.scale_

In [ ]:
adjusted_rows = []
cluster_rows = []

for class_name in RECOVERY_CLASSES:
    group = stars.loc[
        stars["eligible"] & stars["class_name"].eq(class_name)
    ].copy()

    x = recovery_design(group)
    complete = x.notna().all(axis=1)

    group = group.loc[complete]
    x = x.loc[complete]
    target = group["period_recovered"].astype(int).to_numpy()

    if min(np.bincount(target, minlength=2)) < 15:
        continue

    cells = list(group.groupby("sky_cell").indices.values())

    if len(cells) < 8:
        continue

    coefficients = fit_recovery(x, target)
    rng = np.random.default_rng(SEED + RECOVERY_CLASSES.index(class_name))
    draws = []

    for _ in range(CLUSTER_DRAWS):
        selected = rng.integers(len(cells), size=len(cells))
        indices = np.concatenate([cells[i] for i in selected])
        sampled = target[indices]

        if np.unique(sampled).size < 2:
            continue

        draws.append(fit_recovery(x.iloc[indices], sampled))

    cluster_rows.append({
        "class_name": class_name, "n_stars": len(group),
        "n_cells": len(cells), "valid_draws": len(draws),
    })

    if len(draws) < 40:
        continue

    low, high = np.quantile(np.asarray(draws), [0.025, 0.975], axis=0)

    for i, predictor in enumerate(x.columns):
        adjusted_rows.append({
            "class_name": class_name, "predictor": predictor,
            "n_stars": len(group), "n_cells": len(cells),
            "odds_ratio": float(np.exp(coefficients[i])),
            "ci_lo": float(np.exp(low[i])),
            "ci_hi": float(np.exp(high[i])),
        })

adjusted = pd.DataFrame(adjusted_rows)
clusters = pd.DataFrame(cluster_rows)

adjusted.to_csv(EXTRA_ROOT / "adjusted.csv", index=False)
clusters.to_csv(EXTRA_ROOT / "clusters.csv", index=False)

display(adjusted.round(3))
display(clusters)

## Latitude Comparison

In [ ]:
def matched_rate(group):
    differences = []
    weights = []

    for _, part in group.groupby(
        ["epoch_bin", "cycle_bin", "amp_bin"], observed=True
    ):
        ecliptic = part.loc[part["lat_group"].eq("ecliptic"), "period_recovered"]
        polar = part.loc[part["lat_group"].eq("polar"), "period_recovered"]

        if ecliptic.empty or polar.empty:
            continue

        differences.append(polar.mean() - ecliptic.mean())
        weights.append(min(len(ecliptic), len(polar)))

    return (
        float(np.average(differences, weights=weights)) if weights else np.nan,
        len(weights),
        int(sum(weights)),
    )

In [ ]:
matching_rows = []

for class_name in CLASSES:
    for region in ["all", "outside"]:
        group = stars.loc[
            stars["eligible"]
            & stars["class_name"].eq(class_name)
            & stars["amplitude_ratio"].gt(0)
            & stars["n_cycles"].gt(0)
            & (
                stars["abs_ecl_lat"].le(30)
                | stars["abs_ecl_lat"].ge(60)
            )
        ].copy()

        if region == "outside":
            group = group.loc[group["field"].eq("outside")].copy()

        if group.empty:
            continue

        group["lat_group"] = np.where(
            group["abs_ecl_lat"].le(30), "ecliptic", "polar"
        )

        group["cycle_bin"] = pd.qcut(
            np.log10(group["n_cycles"]), 2,
            labels=False, duplicates="drop",
        )

        group["amp_bin"] = pd.qcut(
            np.log10(group["amplitude_ratio"]), 2,
            labels=False, duplicates="drop",
        )

        group = group.dropna(subset=["epoch_bin", "cycle_bin", "amp_bin"])

        contrast, strata, pairs = matched_rate(group)

        cells = list(group.groupby("sky_cell").indices.values())
        rng = np.random.default_rng(
            SEED + CLASSES.index(class_name) + 10 * ["all", "outside"].index(region)
        )
        draws = []

        for _ in range(500):
            if not cells:
                break

            selected = rng.integers(len(cells), size=len(cells))
            indices = np.concatenate([cells[i] for i in selected])
            estimate, _, _ = matched_rate(group.iloc[indices])

            if np.isfinite(estimate):
                draws.append(estimate)

        low, high = np.quantile(draws, [0.025, 0.975]) if draws else (np.nan, np.nan)

        matching_rows.append({
            "class_name": class_name, "region": region,
            "ecliptic_stars": int(group["lat_group"].eq("ecliptic").sum()),
            "polar_stars": int(group["lat_group"].eq("polar").sum()),
            "matched_strata": strata, "matched_pairs": pairs,
            "polar_minus_ecliptic": contrast,
            "ci_lo": low, "ci_hi": high,
        })

matched = pd.DataFrame(matching_rows)
matched.to_csv(EXTRA_ROOT / "matched.csv", index=False)

display(matched.round(3))

## Feature Importance

In [ ]:
importance_folds = pd.DataFrame(importance_rows)

importance = (
    importance_folds.groupby("feature", as_index=False)
    .agg(
        f1_drop=("macro_f1_drop", "mean"),
        f1_std=("macro_f1_drop", "std"),
    )
    .sort_values("f1_drop", ascending=False)
)

feature_groups = {
    col: name for name, cols in GROUPS.items() for col in cols
}

importance["group"] = importance["feature"].map(feature_groups)

importance_folds.to_csv(EXTRA_ROOT / "importance_folds.csv", index=False)
importance.to_csv(EXTRA_ROOT / "importance.csv", index=False)

display(importance.head(15).round(4))

In [ ]:
top = importance.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(top["feature"], top["f1_drop"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set(
    xlabel="Decrease in held-out macro F1",
    ylabel="Feature",
    title="HGB permutation importance",
)

fig.tight_layout()
fig.savefig(PLOT_ROOT / "importance.png", dpi=160, bbox_inches="tight")
plt.show()

## Confusion Matrix

In [ ]:
first = oof.loc[oof["repeat"].eq(1)]

cm = confusion_matrix(
    first["class_name"], first["pred_HGB_full"], labels=CLASSES
)

rates = cm / cm.sum(axis=1, keepdims=True)

display(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(rates, vmin=0, vmax=1, cmap="Blues")

ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=30, ha="right")
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set(
    xlabel="Predicted class",
    ylabel="Gaia class",
    title="HGB out-of-fold confusion matrix",
)

for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(
            j, i, f"{rates[i, j]:.2f}\n({cm[i, j]})",
            ha="center", va="center",
            color="white" if rates[i, j] > 0.5 else "black",
        )

fig.colorbar(image, ax=ax, label="Recall fraction")
fig.tight_layout()
fig.savefig(PLOT_ROOT / "confusion.png", dpi=160, bbox_inches="tight")
plt.show()

## Final Model

In [ ]:
final_model = make_model("HGB", "full")
final_model.fit(stars[BASE_FEATURES], y)

joblib.dump(final_model, MODEL_FILE)

metadata = {
    "model": "HGB",
    "sources": len(stars),
    "folds": FOLDS,
    "repeats": REPEATS,
    "classes": CLASSES,
    "features": BASE_FEATURES,
    "groups": GROUPS,
    "sky_group": "Gaia feature-engineering sky cells",
    "period_reference": "Gaia DR3 published period",
    "recovery_rule": "phase drift <= 0.25 cycles",
    "field_cones": {
        "LMC": LMC_RADIUS,
        "SMC": SMC_RADIUS,
    },
}

with (EXTRA_ROOT / "metadata.json").open("w") as file:
    json.dump(metadata, file, indent=2)

print("Model:", MODEL_FILE)
print("CV scores:", CV_FILE)
print("CV ranking:", RANK_FILE)
print("OOF predictions:", PRED_FILE)
print("Results:", EXTRA_ROOT)
print("Figures:", PLOT_ROOT)